# Kaggle from probability

Realiza los cortes y genera los submits en kaggle a partir del archivo de probabilidades indicado.

## 1) Librerías

In [1]:
suppressMessages({
  require("data.table")
})


## 2) Configuración — **AJUSTAR ESTO A TU ENTORNO**

- `carpeta_experimento`: la ruta real donde están tus `.csv`, tu `results_summary_<experimento>.txt` y donde `kaggle_submit()` creaba la subcarpeta `kaggle/`.
- `experimento`: el número que aparece en el nombre de archivo (`KA9105_...`).
- `dry_run <- TRUE`: mientras esté en `TRUE`, el loop principal solo **muestra** qué comando ejecutaría, sin subir nada de verdad. Pasalo a `FALSE` recién cuando ya revisaste el listado.

In [ ]:
PARAM <- list()
PARAM$experimento <- "AVG_PROBABILITY" # <-- AJUSTAR si hace falta
PARAM$archivo <- "/content/buckets/b1/exp/FINAL/final_prob.txt"   # <-- AJUSTAR
# Si TRUE: solo MUESTRA que haria, no ejecuta ningun submit real.
PARAM$dry_run <- TRUE

PARAM$kaggle_competencia <- "utn-2026-virtual-jr"
PARAM$cortes <- seq(1800, 2400, by = 100)

## 3) Funciones

### 3.1 Submit 

In [ ]:
kaggle_submit_simple <- function(experiment_name, tb_prediccion, cortes, competencia) {

  setorder(tb_prediccion, -prob)
  dir.create("kaggle", showWarnings = FALSE)

  for (envios in PARAM$cortes) {

      cat(sprintf("\n--- Cutoff: %d ---\n", envios))
      cat(sprintf("  Min prob in selection: %.6f\n", tb_prediccion[envios, prob]))
      cat(sprintf("  Max prob overall:      %.6f\n", tb_prediccion[1, prob]))

    tb_prediccion[, Predicted := 0L]
    tb_prediccion[1:envios, Predicted := 1L]

    archivo_kaggle <- sprintf("./kaggle/%s-%d.csv", experiment_name, envios)

    fwrite(
      tb_prediccion[, list(numero_de_cliente, Predicted)],
      file = archivo_kaggle,
      sep  = ","
    )
    cat("File written:", archivo_kaggle, "\n")

    kaggle  <- file.path(Sys.getenv("HOME"), ".venv", "bin", "kaggle")
    comando <- paste(shQuote(kaggle), "competitions submit")
    linea   <- paste(
      comando,
      "-c", competencia,
      "-f", archivo_kaggle,
      "-m", shQuote(sprintf("experiment=%s envios=%d", experiment_name, envios))
    )

    cat("Submitting:", envios, "clients...\n")
    if (PARAM$dry_run) {
      cat("Dry run enabled, skipping command...\n")
    } else {
      salida <- system(linea, intern = TRUE)
      cat(salida, "\n")
    }

    flush.console()
    Sys.sleep(30)
  }

  cat("All submissions done!\n")
}

## 4) Loop principal — el que hace los submits

**Importante:** con `dry_run <- TRUE` (celda 2) esta celda no sube nada, solo imprime los comandos que ejecutaría. Revisalo antes de poner `dry_run <- FALSE` y volver a correr esta celda.

In [ ]:
tb_prediccion <- fread(
  PARAM$archivo,
  sep = "\t",
  colClasses = c(numero_de_cliente = "character")
)

kaggle_submit_simple(
  experiment_name = PARAM$experimento,
  tb_prediccion   = tb_prediccion,
  cortes          = PARAM$cortes,
  competencia     = PARAM$kaggle_competencia
)